# 02 Gold - Sales star schema

**Silver → Gold.** Reads **AdventureWorksSilver** and writes a sales **star schema** to **AdventureWorksGold**,
schema `dbo`. Kenseme generates its ontology from these tables, and the Data Agents query them.

```
                     DimDate              DimSalesChannel
                        │                        │
  DimCustomer ──── FactSalesOrder ───────────────┘          one row per order (header, totals)
                        │
                 FactSalesOrderLine ──── DimProduct ──── DimProductCategory
                 one row per line         295 products    37 categories in 4 groups

  FactSalesOrderLineDetail: the same lines, flattened with their order number, customer and order date
```

| Kind | Answers | Tables |
|---|---|---|
| **Facts** | *what happened*: orders, quantities, money | `FactSalesOrder`, `FactSalesOrderLine` |
| **Flat line table** | one row per line with everything about it, for ontologies that bind one table per class (no joins) | `FactSalesOrderLineDetail` |
| **Dimensions** | *who, what, when, how* you slice by | `DimCustomer`, `DimProduct`, `DimProductCategory`, `DimDate`, `DimSalesChannel` |

**Practices used here**
- **Surrogate keys** (`…Key`), deterministic so a re-run gives the same keys; business keys kept next to them
  (`CustomerAlternateKey`, `SalesOrderNumber`).
- **Both channels conformed** into the same facts, told apart by `DimSalesChannel`.
- **Money** as `decimal(19,4)`, with one meaning per column: `LineAmount` = revenue (before tax),
  `TaxAmount`, `LineTotal` = incl. tax, `DiscountAmount` = list price minus selling price.
  Except in `FactSalesOrderLineDetail`, where money is `double` (shown as `float` on the SQL endpoint): the Fabric
  graph loads decimal columns as null.
- **Key columns NOT NULL**, so primary and foreign keys can be declared on the SQL endpoint (last cell).
- **Checks** before and after writing: no line lost in a join, no orphan keys, Gold revenue = Silver revenue.
- Tables named in business language, in `dbo` (Kenseme-created Fabric ontologies read `dbo`).

Run cell by cell, or **Run all** (about 3 minutes after Spark starts).

In [ ]:
# Configuration and helpers
from delta.tables import DeltaTable
from pyspark.sql import functions as F, Window as W
from pyspark.sql.types import DecimalType, StructField, StructType

SILVER_DB = "AdventureWorksSilver.dbo"  # Silver tables, read by name
def lakehouse_path(name):
    """OneLake path of a lakehouse in this workspace, found by name."""
    return notebookutils.lakehouse.get(name)["properties"]["abfsPath"]

GOLD = lakehouse_path("AdventureWorksGold") + "/Tables/dbo"  # Gold tables folder
MONEY = "decimal(19,4)"

def silver(table):
    return spark.table(f"{SILVER_DB}.{table}")

def save(df, name, *keys):
    """Writes a Gold table with NOT NULL key columns, replacing its rows.

    Written to the table folder, not with saveAsTable: the metastore would lowercase the name (dimcustomer), and the
    SQL endpoint and Kenseme would show it that way. The lakehouse discovers the folder as a table.
    The SQL endpoint refuses a primary key on a nullable column. A DataFrame write stores every column as nullable and
    Delta can't make an existing column NOT NULL, so the table is created from a DDL first. New nullable columns at the
    end are added in place (ALTER TABLE ADD COLUMNS). Any other schema difference removes the folder and creates it
    again. Otherwise only the rows are overwritten, which keeps the keys declared on the SQL endpoint.
    """
    path = f"{GOLD}/{name}"
    wanted = StructType([StructField(f.name, f.dataType, f.name not in keys) for f in df.schema])
    if DeltaTable.isDeltaTable(spark, path):
        stored = spark.read.format("delta").load(path).schema
        added = wanted.fields[len(stored.fields):]
        if stored.fields == wanted.fields[:len(stored.fields)] and added:
            spark.sql(f"ALTER TABLE delta.`{path}` ADD COLUMNS ("
                      + ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in added) + ")")
        elif stored != wanted:
            notebookutils.fs.rm(path, True)
    if not DeltaTable.isDeltaTable(spark, path):
        ddl = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" + ("" if f.nullable else " NOT NULL") for f in wanted)
        spark.sql(f"CREATE TABLE delta.`{path}` ({ddl}) USING DELTA")
    df.write.mode("overwrite").format("delta").save(path)
    print(f"saved dbo.{name}: {spark.read.format('delta').load(path).count():,} rows")

online, reseller = silver("online_sales_line"), silver("reseller_order_line")
product, customer = silver("product"), silver("customer")

## 1. Dimensions
### DimSalesChannel
Two rows, so every order says which channel it came through.

In [ ]:
channel = spark.createDataFrame(
    [(1, "Online", "Web store orders from individual customers"),
     (2, "Reseller", "Orders from reseller accounts at negotiated prices")],
    "SalesChannelKey int, SalesChannelName string, SalesChannelDescription string")
save(channel, "DimSalesChannel", "SalesChannelKey")
display(channel)

### DimProductCategory
One row per category, with its group (Bikes, Components, Clothing, Accessories). Keys follow group and name order.

In [ ]:
category = (product.select("Category", "CategoryGroup").distinct()
            .withColumn("ProductCategoryKey", F.row_number().over(W.orderBy("CategoryGroup", "Category")))
            .select("ProductCategoryKey", F.col("Category").alias("ProductCategoryName"),
                    F.col("CategoryGroup").alias("ProductCategoryGroup")))
save(category, "DimProductCategory", "ProductCategoryKey")
display(category.groupBy("ProductCategoryGroup").count().orderBy("ProductCategoryGroup"))

### DimProduct
The source `ProductID` is already a stable integer, so it is the key. Each product points to its category.

In [ ]:
dim_product = (product.join(category, product.Category == category.ProductCategoryName)
               .select(F.col("ProductId").alias("ProductKey"), "ProductName", "ProductModel", "Variant", "Color",
                       "ListPrice", "ProductCategoryKey"))
assert dim_product.count() == product.count(), "product without a category"
save(dim_product, "DimProduct", "ProductKey")
display(dim_product.limit(5))

### DimCustomer
Online customers first, numbered in order of their first purchase, then reseller accounts.
`CustomerAlternateKey` is the business key from Silver (e-mail or `RS-<account>`).
`FirstName` / `LastName` split the online customer's name: the source has one name field. The first word is the first
name, except the two-word first names in the data (*Carol Ann*, *Francisco Javier*). Reseller accounts: NULL.

In [ ]:
TWO_WORD_FIRST_NAMES = ["Carol Ann", "Francisco Javier"]
individual = F.col("CustomerType") == "Individual"
first = F.coalesce(*[F.when(F.col("CustomerName").startswith(n + " "), F.lit(n)) for n in TWO_WORD_FIRST_NAMES],
                   F.substring_index("CustomerName", " ", 1))
dim_customer = (customer
                .withColumn("CustomerKey", F.row_number().over(W.orderBy(
                    F.when(individual, 0).otherwise(1), "FirstOrderDate", "CustomerAlternateKey")))
                .withColumn("FirstName", F.when(individual, first))
                .withColumn("LastName", F.when(individual, F.expr("substring(CustomerName, length(FirstName) + 2)")))
                .select("CustomerKey", "CustomerAlternateKey", "CustomerName", "EmailAddress", "CustomerType",
                        "FirstOrderDate", "FirstName", "LastName"))
assert dim_customer.where(individual & ((F.col("LastName") == "")
                                        | (F.concat_ws(" ", "FirstName", "LastName") != F.col("CustomerName")))).count() == 0, \
    "online customer name not split into first + last name"
save(dim_customer, "DimCustomer", "CustomerKey")
display(dim_customer.groupBy("CustomerType").count())

### DimDate
Every day from the first order to the end of the last order year, so "per month" questions also show empty months.
`DateKey` is `yyyyMMdd` (20210715).

In [ ]:
bounds = online.select("OrderDate").unionByName(reseller.select("OrderDate")).agg(F.min("OrderDate"), F.max("OrderDate")).first()
dates = spark.sql(f"SELECT explode(sequence(to_date('{bounds[0]}'), to_date('{bounds[1].year}-12-31'))) AS Date")
dim_date = dates.select(
    F.date_format("Date", "yyyyMMdd").cast("int").alias("DateKey"), "Date",
    F.year("Date").alias("CalendarYear"), F.quarter("Date").alias("CalendarQuarter"),
    F.month("Date").alias("MonthNumber"), F.date_format("Date", "MMMM").alias("MonthName"),
    F.date_format("Date", "yyyy-MM").alias("YearMonth"), F.dayofmonth("Date").alias("DayOfMonth"),
    F.date_format("Date", "EEEE").alias("DayOfWeekName"),
    F.dayofweek("Date").isin(1, 7).alias("IsWeekend"))
save(dim_date, "DimDate", "DateKey")
print("from", bounds[0], "to", f"{bounds[1].year}-12-31")

## 2. Facts
### Conform the order lines of both channels
Both channels get the same columns. Online lines have a unit price and 8% tax; reseller lines only a line total,
so their unit price is derived and their tax is 0. Every line is priced against the catalog list price.

In [ ]:
list_price = product.select("ProductId", "ProductName", "ListPrice")
online_lines = (online.join(list_price, "ProductName")
                .select("SalesOrderNumber", "LineNumber", "OrderDate", F.col("EmailAddress").alias("CustomerAlternateKey"),
                        F.lit(1).alias("SalesChannelKey"), "ProductId", "Quantity", "UnitPrice", "ListPrice",
                        (F.col("UnitPrice") * F.col("Quantity")).cast(MONEY).alias("LineAmount"), "TaxAmount"))
reseller_lines = (reseller.join(list_price, "ProductId")
                  .select("SalesOrderNumber", "LineNumber", "OrderDate",
                          F.concat(F.lit("RS-"), F.col("ResellerAccountId")).alias("CustomerAlternateKey"),
                          F.lit(2).alias("SalesChannelKey"), "ProductId", "Quantity",
                          (F.col("LineAmount") / F.col("Quantity")).cast(MONEY).alias("UnitPrice"), "ListPrice",
                          "LineAmount", F.lit(0).cast(MONEY).alias("TaxAmount")))
lines = (online_lines.unionByName(reseller_lines)
         .withColumn("SalesOrderKey", F.substring("SalesOrderNumber", 3, 10).cast("int"))
         .join(dim_customer.select("CustomerKey", "CustomerAlternateKey"), "CustomerAlternateKey")
         .withColumn("OrderDateKey", F.date_format("OrderDate", "yyyyMMdd").cast("int")))
assert lines.count() == online.count() + reseller.count(), "order lines lost in a join"
print(f"{lines.count():,} order lines from both channels")

### FactSalesOrderLine
Grain: one row per order line. `SalesOrderLineKey` = order number × 100 + line number (SO43697 line 1 → 4369701).

In [ ]:
fact_line = lines.select(
    (F.col("SalesOrderKey").cast("bigint") * 100 + F.col("LineNumber")).alias("SalesOrderLineKey"),
    "SalesOrderKey", "LineNumber", F.col("ProductId").alias("ProductKey"), "Quantity", "UnitPrice", "ListPrice",
    "LineAmount",
    F.greatest(F.col("ListPrice") * F.col("Quantity") - F.col("LineAmount"), F.lit(0)).cast(MONEY).alias("DiscountAmount"),
    "TaxAmount", (F.col("LineAmount") + F.col("TaxAmount")).cast(MONEY).alias("LineTotal"))
save(fact_line, "FactSalesOrderLine", "SalesOrderLineKey")
display(fact_line.limit(5))

### FactSalesOrder
Grain: one row per order, with its customer, channel, date and totals. `SubTotal` = order revenue (before tax),
`TotalDue` = what the customer pays (incl. tax).

In [ ]:
fact_order = (lines.groupBy("SalesOrderKey", "SalesOrderNumber", "OrderDateKey", "CustomerKey", "SalesChannelKey")
              .agg(F.count("*").alias("LineCount"), F.sum("Quantity").alias("TotalQuantity"),
                   F.sum("LineAmount").cast(MONEY).alias("SubTotal"), F.sum("TaxAmount").cast(MONEY).alias("TaxAmount"))
              .withColumn("TotalDue", (F.col("SubTotal") + F.col("TaxAmount")).cast(MONEY)))
assert fact_order.count() == fact_order.select("SalesOrderKey").distinct().count(), "order with two headers"
save(fact_order.select("SalesOrderKey", "SalesOrderNumber", "OrderDateKey", "CustomerKey", "SalesChannelKey",
                       "LineCount", "TotalQuantity", "SubTotal", "TaxAmount", "TotalDue"), "FactSalesOrder", "SalesOrderKey")
display(fact_order.limit(5))

### FactSalesOrderLineDetail
Same grain and rows as `FactSalesOrderLine`, flattened: all its columns, plus the order's `SalesOrderNumber` and
`CustomerKey` and the `OrderDate` as a real date. For ontologies whose bindings read one table per class (no joins),
such as a *Sale* class keyed on order number + line number. A Delta table, not a view: bindings read lakehouse tables.
- Its primary key is (`SalesOrderNumber`, `LineNumber`): Kenseme takes a class's key from the table's primary key.
  `SalesOrderLineKey` stays as a plain column.
- Money columns are `double`, not decimal: the Fabric graph loads decimal columns as null.

In [ ]:
fact_line_detail = (fact_line
                    .join(lines.select("SalesOrderKey", "LineNumber", "SalesOrderNumber", "CustomerKey", "OrderDate"),
                          ["SalesOrderKey", "LineNumber"])
                    .select(*fact_line.columns, "SalesOrderNumber", "CustomerKey", F.col("OrderDate").cast("date")))
fact_line_detail = fact_line_detail.select(
    *[F.col(f.name).cast("double") if isinstance(f.dataType, DecimalType) else F.col(f.name) for f in fact_line_detail.schema])
assert fact_line_detail.count() == fact_line.count(), "order lines lost or doubled in the detail table"
assert fact_line_detail.select("SalesOrderNumber", "LineNumber").distinct().count() == fact_line.count(), "order number + line not unique"
save(fact_line_detail, "FactSalesOrderLineDetail", "SalesOrderNumber", "LineNumber")
display(fact_line_detail.limit(5))

## 3. Checks on the saved Gold tables
### Row counts and referential integrity
Every foreign key must find its dimension row (the `orphans` column must be 0).

In [ ]:
%%sql
SELECT 'FactSalesOrderLine -> FactSalesOrder' AS relationship, COUNT(*) AS orphans
FROM AdventureWorksGold.dbo.FactSalesOrderLine l LEFT ANTI JOIN AdventureWorksGold.dbo.FactSalesOrder o ON o.SalesOrderKey = l.SalesOrderKey
UNION ALL SELECT 'FactSalesOrderLine -> DimProduct', COUNT(*)
FROM AdventureWorksGold.dbo.FactSalesOrderLine l LEFT ANTI JOIN AdventureWorksGold.dbo.DimProduct p ON p.ProductKey = l.ProductKey
UNION ALL SELECT 'FactSalesOrder -> DimCustomer', COUNT(*)
FROM AdventureWorksGold.dbo.FactSalesOrder o LEFT ANTI JOIN AdventureWorksGold.dbo.DimCustomer c ON c.CustomerKey = o.CustomerKey
UNION ALL SELECT 'FactSalesOrder -> DimDate', COUNT(*)
FROM AdventureWorksGold.dbo.FactSalesOrder o LEFT ANTI JOIN AdventureWorksGold.dbo.DimDate d ON d.DateKey = o.OrderDateKey
UNION ALL SELECT 'FactSalesOrder -> DimSalesChannel', COUNT(*)
FROM AdventureWorksGold.dbo.FactSalesOrder o LEFT ANTI JOIN AdventureWorksGold.dbo.DimSalesChannel s ON s.SalesChannelKey = o.SalesChannelKey
UNION ALL SELECT 'DimProduct -> DimProductCategory', COUNT(*)
FROM AdventureWorksGold.dbo.DimProduct p LEFT ANTI JOIN AdventureWorksGold.dbo.DimProductCategory c ON c.ProductCategoryKey = p.ProductCategoryKey
UNION ALL SELECT 'FactSalesOrderLineDetail -> FactSalesOrderLine', COUNT(*)
FROM AdventureWorksGold.dbo.FactSalesOrderLineDetail d LEFT ANTI JOIN AdventureWorksGold.dbo.FactSalesOrderLine l ON l.SalesOrderLineKey = d.SalesOrderLineKey
UNION ALL SELECT 'FactSalesOrderLine -> FactSalesOrderLineDetail', COUNT(*)
FROM AdventureWorksGold.dbo.FactSalesOrderLine l LEFT ANTI JOIN AdventureWorksGold.dbo.FactSalesOrderLineDetail d ON d.SalesOrderLineKey = l.SalesOrderLineKey
UNION ALL SELECT 'FactSalesOrderLineDetail -> FactSalesOrder (number, customer, date)', COUNT(*)
FROM AdventureWorksGold.dbo.FactSalesOrderLineDetail d LEFT ANTI JOIN AdventureWorksGold.dbo.FactSalesOrder o
  ON o.SalesOrderKey = d.SalesOrderKey AND o.SalesOrderNumber = d.SalesOrderNumber AND o.CustomerKey = d.CustomerKey
  AND o.OrderDateKey = CAST(date_format(d.OrderDate, 'yyyyMMdd') AS INT)

### Silver vs Gold: nothing lost, nothing added
Lines and revenue per channel must be identical in both layers. Expected total revenue: **21,636,712.50**.

In [ ]:
%%sql
WITH silver AS (
  SELECT 'Online' AS channel, COUNT(*) AS lines, ROUND(SUM(UnitPrice * Quantity), 2) AS revenue
  FROM AdventureWorksSilver.dbo.online_sales_line
  UNION ALL
  SELECT 'Reseller', COUNT(*), ROUND(SUM(LineAmount), 2) FROM AdventureWorksSilver.dbo.reseller_order_line),
gold AS (
  SELECT ch.SalesChannelName AS channel, COUNT(*) AS lines, ROUND(SUM(l.LineAmount), 2) AS revenue
  FROM AdventureWorksGold.dbo.FactSalesOrderLine l
  JOIN AdventureWorksGold.dbo.FactSalesOrder o ON o.SalesOrderKey = l.SalesOrderKey
  JOIN AdventureWorksGold.dbo.DimSalesChannel ch ON ch.SalesChannelKey = o.SalesChannelKey
  GROUP BY ch.SalesChannelName)
SELECT s.channel, s.lines AS silver_lines, g.lines AS gold_lines, s.revenue AS silver_revenue, g.revenue AS gold_revenue,
       CASE WHEN s.lines = g.lines AND s.revenue = g.revenue THEN 'OK' ELSE 'MISMATCH' END AS result
FROM silver s JOIN gold g ON g.channel = s.channel

## 4. Declare the keys (SQL endpoint, once)
Spark can't declare keys. Open the **AdventureWorksGold SQL analytics endpoint** › New SQL query, and run
`Labs/Lab4/Assets/declare_keys_gold.sql` (Lab 4): 8 primary keys and 9 foreign keys, `NOT ENFORCED` (informational). Kenseme
reads them: the ontology generator turns each foreign key into a relationship, and the binding uses them.

```sql
ALTER TABLE dbo.DimCustomer ADD CONSTRAINT PK_DimCustomer PRIMARY KEY NONCLUSTERED (CustomerKey) NOT ENFORCED;
...
ALTER TABLE dbo.FactSalesOrder ADD CONSTRAINT FK_FactSalesOrder_DimCustomer
    FOREIGN KEY (CustomerKey) REFERENCES dbo.DimCustomer (CustomerKey) NOT ENFORCED;
```

Re-running this notebook keeps the keys (only the rows are overwritten). Next: **03 Gold - Business questions**.